In [5]:
# -----------------------
# Step 1. 환경설정
# -----------------------
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from openai import OpenAI

# .env에서 키 불러오기
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("✅ OpenAI API Key 불러오기 성공:", OPENAI_API_KEY[:10] + "...")

# OpenAI embedding 함수 연결
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small"
)

✅ OpenAI API Key 불러오기 성공: sk-proj-IG...


In [6]:

# ChromaDB 클라이언트 생성
client = chromadb.PersistentClient(path="chroma_data")

# -----------------------
# Step 2. 컬럼 메타데이터 로드
# -----------------------
with open("data/meta_json/lw_ded_columns.json", "r", encoding="utf-8") as f:
    columns = json.load(f)

print("✅ JSON 로드 성공. 총 컬럼 수:", len(columns))

✅ JSON 로드 성공. 총 컬럼 수: 28


In [7]:
# =======================================
# 3. JSON 메타데이터 로드
# =======================================
with open("data/meta_json/lw_ded_columns.json", "r", encoding="utf-8") as f:
    columns = json.load(f)
# -----------------------
# Step 3. ChromaDB 컬렉션 생성/가져오기
# -----------------------
collection = client.get_or_create_collection(
    name="lw_ded_columns",
    embedding_function=openai_ef
)

print("✅ Collection 준비 완료:", collection.name)

print(f"✅ Step 3: JSON 로드 완료 (총 {len(columns)}개 컬럼)")
print("샘플 미리보기:", columns[:2])


✅ Collection 준비 완료: lw_ded_columns
✅ Step 3: JSON 로드 완료 (총 28개 컬럼)
샘플 미리보기: [{'column': 'process_id', 'name': '공정 식별자', 'description': '해당 LW-DED 데이터의 고유 번호로, 추적 및 관리에 사용.'}, {'column': 'process_datetime', 'name': '공정 시각', 'description': '공정 시작/기록 시각(ISO8601). 시간 기반 분석 및 이벤트 추적에 활용.'}]


In [8]:
# -----------------------
# Step 4. 메타데이터 업로드
# -----------------------
documents = [f"{c['column']} ({c['name']}): {c['description']}" for c in columns]
ids = [f"col_{i}" for i in range(len(columns))]
metadatas = [{"column": c["column"], "name": c["name"]} for c in columns]

collection.upsert(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)

print("✅ Step 4: 메타데이터 ChromaDB 업로드 완료")
print("업로드된 Document 예시:", documents[0])


✅ Step 4: 메타데이터 ChromaDB 업로드 완료
업로드된 Document 예시: process_id (공정 식별자): 해당 LW-DED 데이터의 고유 번호로, 추적 및 관리에 사용.


In [9]:
# -----------------------
# Step 5. RAG 질의 테스트
# -----------------------
query = "mpt와 process stability score의 관계를 알려줘"

results = collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas"]
)

print("\n=== RAG 검색 결과 ===")
for i, doc in enumerate(results["documents"][0]):
    print(f"\n[Result {i+1}]")
    print(doc[:200], "...")
    print("Source:", results["metadatas"][0][i])


=== RAG 검색 결과 ===

[Result 1]
process_stability_score (Process Stability Score): 온도, 하중 안정성을 바탕으로 한 종합 안정도 평가 점수. ...
Source: {'column': 'process_stability_score', 'name': 'Process Stability Score'}

[Result 2]
mpt (Melt Pool Temperature): 용융풀의 온도. 공정 안정성과 품질을 좌우하는 핵심 변수. ...
Source: {'column': 'mpt', 'name': 'Melt Pool Temperature'}

[Result 3]
process_similarity_score (Process Similarity Score): 타 공정 대비 유사성 지표. 과거 데이터와 비교해 품질 수준을 정량화. ...
Source: {'column': 'process_similarity_score', 'name': 'Process Similarity Score'}


In [10]:
# -----------------------
# Step 6. 데이터 저장 여부 확인
# -----------------------
print("\n=== Step 6. 저장된 Document 개수 확인 ===")
print("총 개수:", collection.count())

sample = collection.get(ids=["col_0"])
print("예시 Document:", sample["documents"][0])
print("예시 Metadata:", sample["metadatas"][0])


=== Step 6. 저장된 Document 개수 확인 ===
총 개수: 28
예시 Document: process_id (공정 식별자): 해당 LW-DED 데이터의 고유 번호로, 추적 및 관리에 사용.
예시 Metadata: {'column': 'process_id', 'name': '공정 식별자'}


In [11]:

# -----------------------
# Step 7. OpenAI LLM과 연결된 RAG 파이프라인
# -----------------------
from openai import OpenAI
client_openai = OpenAI(api_key=OPENAI_API_KEY)

def rag_query(question: str, top_k: int = 3):
    # 1) ChromaDB 검색
    results = collection.query(
        query_texts=[question],
        n_results=top_k,
        include=["documents", "metadatas"]
    )

    retrieved_docs = results["documents"][0]
    context = "\n".join(retrieved_docs)

    # 2) LLM에 질의
    prompt = f"""
    You are an expert in LW-DED process monitoring.
    Answer the following question using the context below.

    Question: {question}

    Context:
    {context}

    Answer:"""

    response = client_openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

In [12]:

# -----------------------
# Step 8. RAG 질의 테스트 실행
# -----------------------
question = "LW-DED에서 mpt(Melt Pool Temperature)와 load 변수는 어떤 의미를 가지나요?"
answer = rag_query(question)

print("\n=== Step 8. RAG 응답 ===")
print("질문:", question)
print("응답:", answer)


=== Step 8. RAG 응답 ===
질문: LW-DED에서 mpt(Melt Pool Temperature)와 load 변수는 어떤 의미를 가지나요?
응답: LW-DED(Load-Wire Directed Energy Deposition)에서 mpt(Melt Pool Temperature)와 load 변수는 다음과 같은 의미를 가집니다.

- **mpt (Melt Pool Temperature)**: 용융풀의 온도를 나타내며, 이는 공정의 안정성과 최종 품질에 매우 중요한 변수입니다. 높은 mpt는 재료의 용융 상태를 최적화하여 접합력을 향상시킬 수 있지만, 과도한 온도는 기계적 성질에 부정적인 영향을 줄 수 있습니다. 따라서 mpt를 적절히 유지하는 것이 LW-DED 공정의 성공에 필수적입니다.

- **load**: LW-DED에서 load 변수는 일반적으로 공정에서 사용되는 재료의 양이나 공급 속도를 의미합니다. 이 변수는 용융풀의 형성과 사이즈, 그리고 최종 출력물의 품질에 직접적인 영향을 미치며, 적절한 load 설정이 없으면 용융풀의 상태가 불안정해질 수 있습니다.

따라서, mpt와 load는 LW-DED 공정의 결과물 품질 및 안정성을 유지하기 위해 모두 중요합니다.


In [13]:
# -----------------------
# Step 1: 환경 변수 로드
# -----------------------
import os
from dotenv import load_dotenv
import chromadb
from chromadb.utils import embedding_functions

# .env 파일 로드
load_dotenv()

# 환경 변수 불러오기
openai_api_key = os.getenv("OPENAI_API_KEY")
chroma_path = os.getenv("CHROMA_DB_PATH", "./chroma_db")

# -----------------------
# Step 2: ChromaDB 연결
# -----------------------
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=openai_api_key,
    model_name="text-embedding-3-small"
)

client = chromadb.PersistentClient(path=chroma_path)

# -----------------------
# Step 3: Collection 리스트 확인
# -----------------------
collections = client.list_collections()

print("=== 현재 ChromaDB Collections ===")
for col in collections:
    print(f"- {col.name}")


=== 현재 ChromaDB Collections ===
- process_meta


In [14]:
# -----------------------
# Step 4: process_meta 컬렉션 접근
# -----------------------
collection = client.get_collection(
    name="process_meta",
    embedding_function=openai_ef
)

# -----------------------
# Step 5: 데이터 확인
# -----------------------
# 문서와 메타데이터 일부만 가져오기 (limit=5)
results = collection.get(
    include=["documents", "metadatas"],
    limit=5
)

print("=== process_meta Collection 내용 ===")
for i, doc in enumerate(results["documents"]):
    print(f"\n[Document {i+1}]")
    print("내용:", str(doc)[:300], "...")   # 문서 앞부분만 출력
    print("Metadata:", results["metadatas"][i])


=== process_meta Collection 내용 ===
